# NFDI4Objects KG — Poseidon aDNA samples: country × capture type

This notebook queries the [NFDI4Objects Knowledge Graph](https://graph.nfdi4objects.net/)
for archaeogenetic **aDNA samples** from the
[Poseidon Community Archive](https://www.poseidon-adna.org/) — loaded
into the N4O KG as `collection/17` under the
[ArNO ontology](https://github.com/archaeonatural-cloud/archaeonatural-ontology)
(Straten, Strohm, Thiery & Renz 2025) — and cross-tabulates them by
**country** and **capture type**. Two complementary views render
the result: a stacked bar chart showing both the overall sample
volume per country and the methodological mix within each, and a
heatmap that makes every country × method combination individually
inspectable.

Unlike the companion notebook `n4okg-poseidon-sites-map.ipynb`,
which asks *where are the samples*, this notebook asks *how were
they produced* — and whether the answer depends on *where they come
from*. It is part of an Open Educational Resource series on
knowledge graphs and linked open data, and is the local Jupyter
companion to `n4okg-poseidon-country-capture-live.qmd`.

## Requirements

```bash
pip install SPARQLWrapper pandas matplotlib
```


## About this notebook

### Why this dataset?

`collection/17` holds the Poseidon Community Archive (PCA), a large
open corpus of archaeogenetic genotype data with contextual and
bibliographic metadata (Schmid et al. 2024). The metadata contains
a field describing the **capture type** of each sample — the
laboratory protocol used to enrich aDNA fragments from a sample
(e.g. *Shotgun*, *1240K*, various targeted panels). This is a
methodological choice of the original project, not an underlying
property of the archaeological record, and its distribution across
countries is therefore an interesting question in its own right.

### What you'll learn

- how a single SPARQL query returning a tidy three-column + one-
  measure table can drive two different visualisations
- how to `pivot_table` a long-format SPARQL result into a
  country × method matrix with `pandas`
- how a **stacked bar** and a **heatmap** answer related but
  distinct questions about the same cross-tabulation

### Data-context notes

- The query traverses the same ArNO path as the sites map
  (`aDNASample → DiscoverySite → Site → Place → Country`), but
  adds a second hop: `aDNASample → hasCaptureType → CaptureType`.
  Each sample contributes to exactly one country–method cell.
- Patterns in the resulting cross-tabulation should not
  automatically be read as archaeological facts. They may reflect
  **publication standards**, **project traditions**, or
  **technical preferences** in particular labs and countries.

### Tooling notes

This local variant uses `SPARQLWrapper` for the query, `pandas`
for the DataFrame and pivot, and `matplotlib` for the two plots.


## Step 1 — Define the SPARQL query

The query selects every `aDNASample` in the graph together with
the country it was discovered in (via the chain
`foundAtDiscoverySite → atSite → atPlace → inCountry`) and its
capture type (via a direct `hasCaptureType` edge).
`COUNT(DISTINCT ?sample)` makes sure we count each sample once,
even if the graph exposes multiple paths. The two labels are
wrapped in `OPTIONAL` so that samples with a missing country or
capture-type label are still represented.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"
USER_AGENT = "OER-Poseidon-CountryCapture-Notebook/1.0"

SPARQL_QUERY = """
PREFIX arno: <http://archaeonatural.cloud/ont/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?countryLabel ?captureTypeLabel (COUNT(DISTINCT ?sample) AS ?numSamples)
WHERE {
  GRAPH <https://graph.nfdi4objects.net/collection/17> {
    ?sample rdf:type arno:aDNASample ;
            arno:foundAtDiscoverySite ?discoverySite ;
            arno:hasCaptureType ?captureType .
    ?discoverySite arno:atSite ?site .
    ?site arno:atPlace ?place .
    ?place arno:inCountry ?country .
    OPTIONAL { ?country rdfs:label ?countryLabel . }
    OPTIONAL { ?captureType rdfs:label ?captureTypeLabel . }
  }
}
GROUP BY ?countryLabel ?captureTypeLabel
ORDER BY DESC(?numSamples)
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
sparql.setMethod("POST")   # long query with GRAPH clause — use POST
sparql.setQuery(SPARQL_QUERY)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
bindings = results["results"]["bindings"]
print(f"✓ Retrieved {len(bindings)} rows from the N4O KG")

## Step 2 — Load the data

The query result is a long-format table: one row per `(country,
capture type)` combination with the sample count attached.


In [ ]:
import pandas as pd

assert bindings, "⚠ Please run the query cell first."

rows = [{
    "country":     r.get("countryLabel",     {}).get("value", "(unknown)"),
    "captureType": r.get("captureTypeLabel", {}).get("value", "(unknown)"),
    "numSamples":  int(r["numSamples"]["value"]),
} for r in bindings]

df = pd.DataFrame(rows)
assert not df.empty, "⚠ No records returned — check query."
print(f"✓ {len(df)} country × capture-type cells "
      f"({df['country'].nunique()} countries, "
      f"{df['captureType'].nunique()} capture types, "
      f"total samples: {df['numSamples'].sum()})")
df.head(10)

## Step 3 — Build the pivot table

The pivot turns the long-format result into a **country × method
matrix** with sample counts as cell values. Missing combinations
are filled with zero: the graph does not know about them, and
leaving the cell as `NaN` would make the bar chart and heatmap
misbehave later.


In [ ]:
assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

pivot = (df.pivot_table(
            index="country", columns="captureType",
            values="numSamples", aggfunc="sum", fill_value=0)
           .astype(int))

# Sort countries by total sample count; capture types by popularity.
country_order = pivot.sum(axis=1).sort_values(ascending=False).index
method_order  = pivot.sum(axis=0).sort_values(ascending=False).index
pivot = pivot.loc[country_order, method_order]

print(f"Pivot shape: {pivot.shape[0]} countries × {pivot.shape[1]} capture types")
print(f"Top 5 countries by total samples:")
for c, n in pivot.sum(axis=1).head(5).items():
    print(f"  {c:25s}  {n}")
print(f"Capture-type totals:")
for m, n in pivot.sum(axis=0).items():
    print(f"  {m:25s}  {n}")
pivot.head()

## Step 4a — Stacked bar chart

The stacked bar gives two pieces of information at once: bar
length encodes the **total sample volume** per country, and the
coloured segments encode the **methodological composition** within
each. A country whose bar is dominated by one colour ran (mostly)
one protocol; a country whose bar shows several colours of
comparable size ran a mixed pipeline.


In [ ]:
import matplotlib.pyplot as plt

assert 'pivot' in dir(), "⚠ Please run the pivot cell first."

TOP_N = 20
top_countries = pivot.head(TOP_N)

cmap = plt.get_cmap("tab20")
colours = [cmap(i % 20) for i in range(len(top_countries.columns))]

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(top_countries))))
top_countries.plot(kind="barh", stacked=True, ax=ax, color=colours,
                   edgecolor="white", linewidth=0.4)
ax.invert_yaxis()   # largest country on top
ax.set_xlabel("number of aDNA samples")
ax.set_ylabel("")
ax.set_title(f"Top {len(top_countries)} countries — aDNA samples by capture type")
ax.legend(title="capture type", bbox_to_anchor=(1.02, 1.0), loc="upper left",
          frameon=False, fontsize=8)
ax.grid(axis="x", linestyle=":", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Step 4b — Heatmap

The heatmap answers the complementary question to the bar chart:
for *every* country × method combination, how many samples fall
into that cell? Empty cells render as very pale grey so they are
visually distinct from small-but-present ones. The colour scale
is **logarithmic** because the cell counts are heavily right-
skewed (a few cells in the hundreds, many in the single digits);
linear scaling would collapse most of the matrix into one colour.

Column labels are placed on top of the plot so they don't collide
with the figure edge at rotation.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

assert 'pivot' in dir(), "⚠ Please run the pivot cell first."

TOP_N_ROWS = 25
heat = pivot.head(TOP_N_ROWS)

data = heat.values.astype(float)
positive = data[data > 0]
if positive.size == 0:
    vmin, vmax = 1, 1
else:
    vmin = max(1, positive.min())
    vmax = positive.max()

display_data = np.where(data > 0, data, vmin / 10.0)

fig, ax = plt.subplots(figsize=(max(7.5, 1.3 * heat.shape[1] + 4.5),
                                max(5.5, 0.42 * heat.shape[0] + 1.2)))
cmap = plt.get_cmap("YlOrRd").copy()
cmap.set_under("#f1f1f1")
cmap.set_bad("#f1f1f1")

im = ax.imshow(display_data, aspect="auto", cmap=cmap,
               norm=LogNorm(vmin=vmin, vmax=vmax),
               interpolation="nearest")

# Column labels on TOP so they don't get clipped by the figure edge.
ax.set_xticks(range(heat.shape[1]))
ax.set_xticklabels(heat.columns, rotation=30, ha="left", fontsize=9)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.tick_params(axis="x", length=0)

ax.set_yticks(range(heat.shape[0]))
ax.set_yticklabels(heat.index, fontsize=9)
ax.tick_params(axis="y", length=0)

# White hairlines between cells — same trick seaborn heatmaps use.
ax.set_xticks(np.arange(heat.shape[1] + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(heat.shape[0] + 1) - 0.5, minor=True)
ax.grid(which="minor", color="white", linewidth=1.3)
ax.tick_params(which="minor", length=0)

ax.set_title(f"Samples per (country × capture type) — top {len(heat)} countries",
             pad=40, fontsize=11)

# Cell annotations — dark cells get white text, light cells black.
log_mid = (np.log(vmin) + np.log(vmax)) / 2 if vmax > vmin else np.log(vmin)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = int(heat.values[i, j])
        if v > 0:
            text_colour = "white" if np.log(max(v, 1)) > log_mid else "#222"
            ax.text(j, i, f"{v:,}" if v >= 1000 else str(v),
                    ha="center", va="center",
                    fontsize=8, color=text_colour)

cbar = plt.colorbar(im, ax=ax, pad=0.02, extend="min")
cbar.set_label("samples per cell (log scale)")
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

## Step 5 — Exploring the data

Two starting points: the full pivot is available as `pivot`, and
the long-format result is available as `df`. Change the thresholds
and see how the picture shifts.


In [ ]:
assert 'pivot' in dir(), "⚠ Please run the pivot cell first."

# Which countries are most methodologically diverse?
diversity = (pivot > 0).sum(axis=1).sort_values(ascending=False)
print(f"Top 10 countries by number of distinct capture types used:")
for c, n in diversity.head(10).items():
    total = int(pivot.loc[c].sum())
    print(f"  {c:25s}  {n} capture types, {total} samples")

In [ ]:
assert 'pivot' in dir(), "⚠ Please run the pivot cell first."

# For any given country, what's the share of each capture type?
COUNTRY = pivot.index[0]   # most-sampled country by default
row = pivot.loc[COUNTRY]
total = row.sum()
print(f"{COUNTRY} — {total} samples, capture-type shares:")
shares = (row / total * 100).sort_values(ascending=False)
for m, pct in shares.items():
    if row[m] > 0:
        print(f"  {m:25s}  {row[m]:5d}  ({pct:5.1f}%)")

---

## References

- **Straten, M. thor, Strohm, S., Thiery, F. & Renz, M.** (2025). Data-Driven
  Community Standards for Interdisciplinary Heterogeneous Information
  Networks. *E-Science-Tage 2025.* Heidelberg: heiBOOKS.
  doi: [10.11588/heibooks.1652.c23914](https://doi.org/10.11588/heibooks.1652.c23914)
- **Schmid, C. et al.** (2024). Poseidon — A framework for archaeogenetic
  human genotype data management. *eLife* 13.
  doi: [10.7554/eLife.98317.1](https://doi.org/10.7554/eLife.98317.1)
- **archaeonatural-cloud/poseidon2lod** — RDF generation scripts.
  [github.com/archaeonatural-cloud/poseidon2lod](https://github.com/archaeonatural-cloud/poseidon2lod)

*This notebook is part of the Open Educational Resources of
[NFDI4Objects](https://www.nfdi4objects.net/).*
